# Stage 1: Experimental — VWAP-based wallet preselection

Preselect larger wallet sets that are profitable (average_roi > 0.02), then
compute per-trade trailing 15-minute VWAP and VWAP volume for BUY trades
by (wallet, condition_id, token_id).

These VWAP features are added back to the trade DataFrames for downstream
analysis.

**Output:** `stage1_experimental_result.json` with preselected wallets + VWAP stats.

In [669]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from IPython.display import display

from lib import (
    load_trades,
    split_data,
    compute_copyable_notional,
    compute_opening_metrics,
    DEFAULT_TAGS,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Parameters

In [670]:
MIN_ROI = 0.03
VWAP_WINDOW_MINUTES = 15

print(f"MIN_ROI: {MIN_ROI}")
print(f"VWAP_WINDOW_MINUTES: {VWAP_WINDOW_MINUTES}")

MIN_ROI: 0.03
VWAP_WINDOW_MINUTES: 15


## Load data

In [671]:
df_full = load_trades()
df_full = compute_copyable_notional(df_full)
df_train, df_val, df_test = split_data(df_full, method='chronological')

Markets: 1974837
Filtered markets for {'Weather'}: 91123
Loading 16 trade shards...
Total trades loaded: 14,250,603
Unique wallets: 4,082
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-27 06:12:25+00:00
Chronological split: train <= 2026-05-21T00:00:00Z, val <= 2026-06-23T00:00:00Z, test > 2026-06-23T00:00:00Z
Method: chronological  |  Unique end dates: 112  (train=44, val=33, test=35)

  Train:  4,337,961 trades  (15,923 markets)
  Val:    5,211,194 trades  (20,297 markets)
  Test:   4,701,448 trades  (20,655 markets)
  Total: 14,250,603 trades  (56,875 markets)


## Compute wallet metrics on training data

In [672]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi", "num_buckets"]].head(10)

Wallets with metrics: 3220


,wallet,buy_roi,sell_roi,copyable_pnl,copyable_roi,num_buckets
0,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.008757,0.001249,-0.527030,-0.000000,38
1,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.039375,0.139649,2.590081,0.069611,208
2,0x00833cc2d777e6f2fc8437679124024ae6468cb1,-0.070000,0.050000,0.000000,0.000000,2
3,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.028754,-0.010000,57.724739,0.243228,34
4,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,-0.009430,-0.025010,-17.576089,-0.219729,299
5,0x01a5fb1fa13f378138a31382c8364b5d4e2b0e36,0.008435,-0.001600,3.291636,-0.046859,23
6,0x01a68281185e728ba0fef6245008bf8af68a59b0,0.009914,-0.144371,-0.667974,0.000000,844
7,0x01ced860d8dca5d7987579d2a2635df8520d27a2,0.072584,-0.040233,-401.033287,0.000000,989
8,0x01d94480e2a96cdd01fed071878b1adf82e0acd0,0.000142,-0.000888,30.651906,0.000000,1174
9,0x01f2a8baabe17c2541d1e3091220991f257ac3de,0.005061,0.003355,-3381.958420,0.000000,39804


## Preselect wallets by average buy ROI

In [673]:
preselected_ws = set(
    wallet_vol.loc[wallet_vol["buy_roi"] > MIN_ROI, "wallet"]
)
print(f"Preselected wallets (buy_roi > {MIN_ROI}): {len(preselected_ws)}")

# Quick stats on the preselected set
preselected_df = wallet_vol[wallet_vol["wallet"].isin(preselected_ws)].copy()
print()
print(f"  Buy ROI range:  {preselected_df['buy_roi'].min():.4f} — {preselected_df['buy_roi'].max():.4f}")
print(f"  Avg num_buckets: {preselected_df['num_buckets'].mean():.0f}")
print(f"  total_pnl: ${preselected_df['total_pnl'].sum():,.0f}")
print(f"  trades: {preselected_df['trade_count'].sum():,.0f}")

Preselected wallets (buy_roi > 0.03): 1142

  Buy ROI range:  0.0300 — 0.9868
  Avg num_buckets: 170
  total_pnl: $653,571
  trades: 298,841


## Compute 15-min trailing VWAP for BUY trades

For each BUY trade by a preselected wallet, look back 15 minutes over
the same (wallet, condition_id, token_id) and compute:
- **vwap_15m**: volume-weighted average price of trades **strictly before** this one
- **vwap_volume_15m**: total USDC volume of trades in the window

> `TEST_MODE=True` limits to 50 wallets for quick validation.

In [674]:
# buy_mask = df_full["wallet"].isin(preselected_ws) & (df_full["side"] == "BUY") 
# buy_trades = df_full.loc[buy_mask].copy() 
# print(f"BUY trades by preselected wallets: {len(buy_trades):,}")

# buy_trades = buy_trades.sort_values(
#     ["wallet", "condition_id", "token_id", "dt"],
#     kind="mergesort"
# ).reset_index(drop=True)

# buy_trades["vwap_15m"] = np.nan
# buy_trades["vwap_volume_15m"] = 0.0

# window_ns = np.timedelta64(VWAP_WINDOW_MINUTES, "m")

# for _, idx in buy_trades.groupby(
#     ["wallet", "condition_id", "token_id"],
#     sort=False
# ).groups.items():

#     g = buy_trades.loc[idx]

#     t = g["dt"].values
#     qty = g["quantity"].to_numpy()
#     usdc = g["usdc_amount"].to_numpy()
#     pq = (g["price"] * g["quantity"]).to_numpy()

#     left = 0
#     sum_qty = 0.0
#     sum_pq = 0.0
#     sum_usdc = 0.0

#     out_vwap = np.empty(len(g))
#     out_vol = np.empty(len(g))

#     for i in range(len(g)):

#         # remove expired trades
#         while left < i and t[left] < t[i] - window_ns:
#             sum_qty -= qty[left]
#             sum_pq -= pq[left]
#             sum_usdc -= usdc[left]
#             left += 1

#         # current trade is excluded
#         out_vwap[i] = np.nan if sum_qty == 0 else sum_pq / sum_qty
#         out_vol[i] = sum_usdc

#         # add current trade
#         sum_qty += qty[i]
#         sum_pq += pq[i]
#         sum_usdc += usdc[i]

#     buy_trades.loc[idx, "vwap_15m"] = out_vwap
#     buy_trades.loc[idx, "vwap_volume_15m"] = out_vol

# df_full = df_full.merge(
#     buy_trades[["tx_hash","vwap_15m","vwap_volume_15m"]],
#     on="tx_hash",
#     how="left",
# )

# df_train = df_full[df_full["dt"] < train_cutoff].copy()
# df_val = df_full[(df_full["dt"] >= train_cutoff) & (df_full["dt"] < val_cutoff)].copy()
# df_test = df_full[df_full["dt"] >= val_cutoff].copy()

In [675]:
bad_buy_leaders = wallet_vol[
    (wallet_vol['buy_roi'] >= 0.03)
    & (wallet_vol['trade_count'] >= 100)
    & (wallet_vol['total_pnl'] > 1000)
    & (wallet_vol['max_drawdown_to_pnl'] <= 0.3)
    & (wallet_vol['buy_copyable_pnl'] < 0)
]

print(f"Bad buy leaders: {len(bad_buy_leaders)}")
print(f"Bad buy leader train trades: {len(df_train[df_train['wallet'].isin(bad_buy_leaders['wallet'])])}")
print(f"Bad buy leader pnl: ${bad_buy_leaders['total_pnl'].sum():,.0f}")
print(f"Bad buy leader copyable buy pnl: ${bad_buy_leaders['buy_copyable_pnl'].sum():,.0f}")
print(f"Bad buy leader buy roi: ${bad_buy_leaders['buy_pnl'].sum() / bad_buy_leaders['buy_notional'].sum():,.2f}")
print(f"Bad buy leader val pnl: ${df_val[df_val['wallet'].isin(bad_buy_leaders['wallet'])]['pnl'].sum():,.0f}")

Bad buy leaders: 9
Bad buy leader train trades: 16137
Bad buy leader pnl: $40,646
Bad buy leader copyable buy pnl: $-2,681
Bad buy leader buy roi: $0.27
Bad buy leader val pnl: $14,626


In [ ]:
class GroupSignal:
    def __init__(self, name, mask, col, side):
        self.name = name
        self.mask = mask
        self.col = col
        self.side = side

In [772]:
selected_filters = [
    GroupSignal(
        name='bad_leader',
        mask=(wallet_vol['buy_roi'] >= 0.03)
            & (wallet_vol['trade_count'] >= 100)
            & (wallet_vol['total_pnl'] > 1000)
            & (wallet_vol['max_drawdown_to_pnl'] <= 0.3)
            & (wallet_vol['buy_copyable_pnl'] < 0),
        col='wallet',
    ),
    GroupSignal(
        name='top_market_maker',
        mask=(wallet_vol['buy_roi'] >= 0.03)
            & (wallet_vol['trade_count'] >= 5000)
            & (wallet_vol['total_pnl'] > 1000)
            & (wallet_vol['max_drawdown_to_pnl'] <= 0.3)
            & (wallet_vol['buy_copyable_pnl'] >= 0),
        col='position',
    ),
    GroupSignal(
        name='wider_market_maker',
        mask=( (wallet_vol['trade_count'] >= 500)
            & (wallet_vol['total_pnl'] > 3000)
            & (wallet_vol['max_drawdown_to_pnl'] <= 0.3)
            & (wallet_vol['buy_copyable_pnl'] < 100)
        ),
        col='position',
    ),
    GroupSignal(
            name='bad_copy_example',
            mask=( (wallet_vol['trade_count'] >= 500)
                & (wallet_vol['total_pnl'] > 3000)
                & (wallet_vol['buy_copyable_pnl'] < -500)
            ),
            col='wallet',
        ),
    GroupSignal(
            name='good_predictor',
            mask=( (wallet_vol['buy_roi'] >= 0.07)
                & (wallet_vol['trade_count'] >= 100)
                & (wallet_vol['total_pnl'] > 1000)
            ),
            col='wallet',
        ),
]

selected_filter_names = [name for f in selected_filters for name in (f.name, f"{f.name}_opposite")]

for signal in selected_filters:
    name, filt, col = signal.name, signal.mask, signal.col
    selected = wallet_vol[filt]
    print(f"  {name}: {len(selected)} wallets")
    print(f"  {name}: total_pnl: ${selected['total_pnl'].sum():,.0f}")
    print(f"  {name}: roi: {selected['total_pnl'].sum() / selected['total_notional'].sum() :,.4f}")
    print(f"  {name}: trade_count: {selected['trade_count'].sum():,.0f}")
    print(f"  {name}: buy_copyable_pnl: ${selected['buy_copyable_pnl'].sum():,.0f}")
    print(f"  {name}: buy_copyable_roi: {selected['buy_copyable_pnl'].sum() / selected['buy_copyable_notional'].sum() :,.4f}")

    print()


  bad_leader: 9 wallets
  bad_leader: total_pnl: $40,646
  bad_leader: roi: 0.1104
  bad_leader: trade_count: 16,137
  bad_leader: buy_copyable_pnl: $-2,681
  bad_leader: buy_copyable_roi: -0.1028

  top_market_maker: 3 wallets
  top_market_maker: total_pnl: $26,256
  top_market_maker: roi: 0.0920
  top_market_maker: trade_count: 40,629
  top_market_maker: buy_copyable_pnl: $1,571
  top_market_maker: buy_copyable_roi: 0.0702

  wider_market_maker: 11 wallets
  wider_market_maker: total_pnl: $82,518
  wider_market_maker: roi: 0.0214
  wider_market_maker: trade_count: 438,194
  wider_market_maker: buy_copyable_pnl: $-13,645
  wider_market_maker: buy_copyable_roi: -0.0391

  bad_copy_example: 11 wallets
  bad_copy_example: total_pnl: $85,102
  bad_copy_example: roi: 0.0199
  bad_copy_example: trade_count: 739,858
  bad_copy_example: buy_copyable_pnl: $-25,045
  bad_copy_example: buy_copyable_roi: -0.0380

  good_predictor: 51 wallets
  good_predictor: total_pnl: $200,463
  good_predictor:

In [773]:
#TODO: split to buy and sell
replace = True

for signal in selected_filters:
    name, filt, col = signal.name, signal.mask, signal.col
    print(f"Processing {name}..." )
    selected = wallet_vol[filt]

    if not replace and name in df_full.columns:
        print("Replace off, skipping")
        continue

    df_full = df_full.drop(columns=[name, f"{name}"], errors='ignore')
    df_full = df_full.drop(columns=[name, f"{name}_opposite"], errors='ignore')

    mask = df_full["wallet"].isin(selected["wallet"])

    selected_trades = df_full[
        df_full["wallet"].isin(selected["wallet"])
         & (df_full["side"] == "BUY")
         ].copy()
    
    selected_trades[name] = selected_trades[col].notna().astype(np.float64)

    print(f"  {name} trades: {len(selected_trades[name])}")

    df_full = pd.merge_asof(
        df_full.sort_values("dt"),
        selected_trades.sort_values("dt")[['dt', name, 'condition_id', 'outcome']],
        on="dt",
        by=["condition_id", "outcome"],
        direction="backward",
        tolerance=pd.Timedelta(minutes=5),
        allow_exact_matches=False,
    )

    selected_trades_opposite = selected_trades.assign(outcome=selected_trades['outcome'].map({'No': 'Yes', 'Yes': 'No'}))
    selected_trades_opposite = selected_trades_opposite.rename(columns={name: f"{name}_opposite"})

    df_full = pd.merge_asof(
            df_full.sort_values("dt"),
            selected_trades_opposite.sort_values("dt")[['dt', f"{name}_opposite", 'condition_id', 'outcome']],
            on="dt",
            by=["condition_id", "outcome"],
            direction="backward",
            tolerance=pd.Timedelta(minutes=5),
            allow_exact_matches=False,
        )

    print(f"Merged {name} and {name}_opposite: {len(selected)} wallets")
    print(f"  total_pnl: ${selected['total_pnl'].sum():,.0f}")
    print(f"  trades: {selected['trade_count'].sum():,.0f}")
    print(f"  buy_copyable_pnl: ${selected['buy_copyable_pnl'].sum():,.0f}")
    

Processing bad_leader...
  bad_leader trades: 9
Merged bad_leader and bad_leader_opposite: 9 wallets
  total_pnl: $40,646
  trades: 16,137
  buy_copyable_pnl: $-2,681
Processing top_market_maker...
  top_market_maker trades: 3
Merged top_market_maker and top_market_maker_opposite: 3 wallets
  total_pnl: $26,256
  trades: 40,629
  buy_copyable_pnl: $1,571
Processing wider_market_maker...
  wider_market_maker trades: 11
Merged wider_market_maker and wider_market_maker_opposite: 11 wallets
  total_pnl: $82,518
  trades: 438,194
  buy_copyable_pnl: $-13,645
Processing bad_copy_example...
  bad_copy_example trades: 11
Merged bad_copy_example and bad_copy_example_opposite: 11 wallets
  total_pnl: $85,102
  trades: 739,858
  buy_copyable_pnl: $-25,045
Processing good_predictor...
  good_predictor trades: 51
Merged good_predictor and good_predictor_opposite: 51 wallets
  total_pnl: $200,463
  trades: 35,818
  buy_copyable_pnl: $27,709


In [774]:
if 'bad_leader_wallet' not in df_full.columns:
    bad_buy_leader_trades = df_full[(df_full['wallet'].isin(bad_buy_leaders['wallet'])) & (df_full['side'] == 'BUY')]
    print(f"Bad buy leader full trades: {len(bad_buy_leader_trades)}")

    leaders = bad_buy_leader_trades.rename(columns={"dt": "dt_leader", "wallet": "bad_leader_wallet"})[['dt_leader', 'bad_leader_wallet', 'condition_id', 'outcome']]

    df_full = pd.merge_asof(
        df_full.sort_values("dt"),
        leaders.sort_values("dt_leader"),
        left_on="dt",
        right_on="dt_leader",
        by=["condition_id", "outcome"],
        direction="backward",
        tolerance=pd.Timedelta(minutes=5),
        allow_exact_matches=False,
)

## Signal Quality Framework

We evaluate each signal using **Information Coefficient (IC)** and
**Information Ratio (IR)**, following Grinold & Kahn (1999),
*Active Portfolio Management* (McGraw-Hill).

| Metric | Definition | Interpretation |
|--------|------------|----------------|
| **IC** | Spearman rank correlation between signal value and forward copyable ROI | Does a higher signal predict better PnL? |
| **IR** | Mean(IC) / Std(IC) across daily chunks | How consistent is the predictive power? |
| **Hit Rate** | % of events where signal sign matches PnL sign | Directional accuracy |
| **Bootstrap CI** | 2.5th-97.5th percentile of IC over 10k resamples | Is IC sign reliably non-zero? |

Signal overlap is measured with **coincidence rate** (do they fire together?)
and **IC correlation** (are their predictions redundant?).

In [775]:

# Signal quality: IC, IR, bootstrap, overlap, combination

import numpy as np
import pandas as pd


def _rankdata(v):
    """Fractional ranking (scipy.stats.rankdata, method='average')."""
    n = len(v)
    sorter = np.argsort(v, kind="mergesort")
    ordinal = np.empty(n, dtype=np.intp)
    ordinal[sorter] = np.arange(n)
    inv = np.argsort(sorter, kind="mergesort")
    rank = ordinal + 1.0
    i = 0
    while i < n:
        j = i + 1
        while j < n and v[sorter[j]] == v[sorter[i]]:
            j += 1
        if j > i + 1:
            avg_rank = (i + j + 1) / 2.0
            for k in range(i, j):
                rank[sorter[k]] = avg_rank
        i = j
    return rank[inv]


def spearman_rho(x, y):
    """Spearman rank correlation (numpy-only)."""
    mask = np.isfinite(x) & np.isfinite(y)
    n = mask.sum()
    if n < 10:
        return np.nan
    rx = _rankdata(x[mask].values if hasattr(x, 'values') else x[mask])
    ry = _rankdata(y[mask].values if hasattr(y, 'values') else y[mask])
    rx_m = rx.mean()
    ry_m = ry.mean()
    num = np.sum((rx - rx_m) * (ry - ry_m))
    den = np.sqrt(np.sum((rx - rx_m)**2) * np.sum((ry - ry_m)**2))
    return num / den if den != 0 else np.nan


def compute_event_ic(signal, forward_roi):
    """IC: rank correlation between signal and forward copyable ROI."""
    return spearman_rho(signal, forward_roi)


def compute_event_ir(signal, forward_roi, timestamps, freq="D"):
    """IR = mean(IC_chunk) / std(IC_chunk) across time chunks.
    
    Higher IR means predictive power is consistent (Grinold & Kahn Ch. 7).
    """
    ts = timestamps
    chunks = pd.Series(index=pd.DatetimeIndex(ts), data=np.arange(len(signal))).groupby(
        pd.Grouper(freq=freq)
    )
    ics = []
    for _, idx in chunks:
        if len(idx) < 5:
            continue
        rho = compute_event_ic(signal.iloc[idx], forward_roi.iloc[idx])
        if not np.isnan(rho):
            ics.append(rho)
    if len(ics) < 3:
        return np.nan
    arr = np.array(ics)
    return float(arr.mean() / arr.std(ddof=1)) if arr.std(ddof=1) > 0 else np.nan


def bootstrap_ic(signal, forward_roi, n_iter=10_000, alpha=0.05, seed=42):
    """Bootstrap CI for IC (Efron & Tibshirani 1993).
    
    Returns (mean_ic, ci_lower, ci_upper).
    """
    mask = signal.notna() & forward_roi.notna()
    s = signal[mask].values
    p = forward_roi[mask].values
    n = len(s)
    if n < 10:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot_ics = np.empty(n_iter)
    for i in range(n_iter):
        idx = rng.integers(0, n, n)
        boot_ics[i] = spearman_rho(s[idx], p[idx])
    mean_ic = float(np.nanmean(boot_ics))
    ci_lo = float(np.nanpercentile(boot_ics, 100 * alpha / 2))
    ci_hi = float(np.nanpercentile(boot_ics, 100 * (1 - alpha / 2)))
    return mean_ic, ci_lo, ci_hi


def hit_rate(signal, forward_roi):
    """Fraction of events where signal sign matches PnL sign."""
    mask = signal.notna() & forward_roi.notna() & (forward_roi != 0)
    if mask.sum() < 10:
        return np.nan
    sgn_sig = np.sign(signal[mask])
    sgn_pnl = np.sign(forward_roi[mask])
    return float((sgn_sig == sgn_pnl).mean())


def signal_quality_report(signals_df, signal_cols, roi_col="copyable_roi",
                           dt_col="dt", ir_freq="D", n_bootstrap=5_000):
    """Compute IC, IR, hit rate, and bootstrap CI for each signal.
    
    Returns DataFrame with one row per signal.
    """
    print(f"Signal quality report: {len(signal_cols)} signals, {len(signals_df)} events")
    rows = []
    for col in signal_cols:
        ic = compute_event_ic(signals_df[col], signals_df[roi_col])

        ir = compute_event_ir(signals_df[col], signals_df[roi_col],
                               signals_df[dt_col], freq=ir_freq)

        hr = hit_rate(signals_df[col], signals_df[roi_col])

        # m, clo, chi = bootstrap_ic(signals_df[col], signals_df[roi_col],
        #                             n_iter=n_bootstrap)
        
        # print(f"  Bootstrap mean IC({col}): {m:.4f}  CI=({clo:.4f}, {chi:.4f})")
        m, clo, chi = -1, -1, -1
        rows.append({
            "signal": col,
            "IC": ic,
            "IR": ir,
            "hit_rate": hr,
            "bootstrap_mean_ic": m,
            "bootstrap_ci_lo": clo,
            "bootstrap_ci_hi": chi,
            "n_events": int(signals_df[col].notna().sum()),
        })
    return pd.DataFrame(rows).sort_values("IC", ascending=False, key=abs)


# === Signal overlap ===

def coincidence_rate(s1, s2):
    """Jaccard-like coincidence: P(both non-zero | either non-zero)."""
    both = ((s1.notna() & (s1 != 0)) & (s2.notna() & (s2 != 0))).sum()
    either = ((s1.notna() & (s1 != 0)) | (s2.notna() & (s2 != 0))).sum()
    return both / either if either > 0 else 0.0


def ic_correlation_matrix(signals_df, signal_cols, roi_col="copyable_roi"):
    """Pairwise IC of signal values on overlapping events."""
    n = len(signal_cols)
    mat = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(n):
            if i == j:
                mat[i, j] = 1.0
                continue
            both = signals_df[signal_cols[i]].notna() & signals_df[signal_cols[j]].notna()
            if both.sum() < 10:
                continue
            mat[i, j] = compute_event_ic(
                signals_df.loc[both, signal_cols[i]],
                signals_df.loc[both, signal_cols[j]],
            )
    return pd.DataFrame(mat, index=signal_cols, columns=signal_cols)


# === Signal combination ===

def compute_optimal_weights(
    signals_df, signal_cols, roi_col="copyable_roi",
    shrinkage=0.5,
):
    """Markowitz-optimal signal weights with shrinkage (Grinold & Kahn Ch. 13).
    
    w = (1-lambda) * inv(Sigma) * IC + lambda * (1/n)
    
    Parameters
    ----------
    shrinkage : float
        0 = full Markowitz, 1 = equal weight.
    
    Returns
    -------
    pd.Series of weights indexed by signal_cols.
    """
    n = len(signal_cols)
    ic_vec = np.array([
        compute_event_ic(signals_df[c], signals_df[roi_col]) or 0.0
        for c in signal_cols
    ])
    
    valid = signals_df[signal_cols].notna().all(axis=1)
    if valid.sum() < 10 or n <= 1:
        return pd.Series(np.ones(n) / n, index=signal_cols)
    
    sig_vals = signals_df.loc[valid, signal_cols].values
    cov = np.cov(sig_vals, rowvar=False)
    avg_var = np.trace(cov) / n
    shrunk_cov = (1 - shrinkage) * cov + shrinkage * np.eye(n) * avg_var
    
    try:
        inv_cov = np.linalg.solve(shrunk_cov, np.eye(n))
        w = inv_cov @ ic_vec
        w_abs_sum = np.sum(np.abs(w))
        if w_abs_sum > 1e-12:
            w = w / w_abs_sum
        else:
            w = np.ones(n) / n
    except np.linalg.LinAlgError:
        w = np.ones(n) / n
    
    return pd.Series(w, index=signal_cols)


def apply_composite_score(signals_df, signal_cols, weights):
    """Composite signal = sum w_i * signal_i."""
    result = np.zeros(len(signals_df))
    for col in signal_cols:
        result += weights[col] * signals_df[col].fillna(0.0).values
    return pd.Series(result, index=signals_df.index)


def cs_rank(s, grouper=None):
    """Cross-sectional rank transform. Maps values to [-1, 1] within groups.
    
    If grouper is provided, ranks within each group independently.
    Standard Grinold & Kahn normalization.
    """
    if grouper is not None:
        result = s.groupby(grouper, sort=False).transform(
            lambda x: 2.0 * (_rankdata(x.values) - 1.0) / max(len(x) - 1, 1) - 1.0
        )
    else:
        n = len(s)
        r = _rankdata(s.values) if hasattr(s, 'values') else _rankdata(np.asarray(s))
        result = 2.0 * (r - 1.0) / max(n - 1, 1) - 1.0
    return result


## Parameters & Test Mode

In [776]:

# Test mode
TEST_MODE = False
MAX_CANDIDATE_WALLETS = 20
MAX_CANDIDATE_TRADES = 5000

# Signal windows (minutes)
BAD_LEADER_WINDOW = 5
QUALITY_WALLET_WINDOW = 15
VWAP_WINDOW = 15

# Quality wallet definition
QW_MIN_BUY_ROI = 0.04
QW_MIN_BUCKETS = 50
QW_MIN_TRADE_COUNT = 1000
QW_MAX_DD_TO_PNL = 0.3
QW_MIN_COPYABLE_ROI = -1

# Copy universe
COPY_MIN_BUY_ROI = 0.02
COPY_MIN_BUCKETS = 20
COPY_MIN_MARKETS = 15
COPY_MIN_TRADE_COUNT = 100
COPY_MAX_DD_TO_PNL = 0.6
COPY_MIN_COPYABLE_ROI = 0.05

print(f"TEST_MODE: {TEST_MODE}")
print(f"Signals: bad_leader ({BAD_LEADER_WINDOW}min), "
      f"quality_wallet ({QUALITY_WALLET_WINDOW}min), "
      f"vwap_deviation ({VWAP_WINDOW}min)")


TEST_MODE: False
Signals: bad_leader (5min), quality_wallet (15min), vwap_deviation (15min)


## Define Copy Universe (candidate trades)

Trades we *could* copy. BUY trades by wallets that pass a quality filter.

In [777]:

# Copy universe: wallets that pass quality + stability filter
copy_mask = (
    (wallet_vol['buy_roi'] >= COPY_MIN_BUY_ROI)
    & (wallet_vol['num_buckets'] >= COPY_MIN_BUCKETS)
    & (wallet_vol['num_markets'] >= COPY_MIN_MARKETS)
    & (wallet_vol['trade_count'] >= COPY_MIN_TRADE_COUNT)
    & (wallet_vol['max_drawdown_to_pnl'].fillna(1.0) <= COPY_MAX_DD_TO_PNL)
    & (wallet_vol['copyable_roi'].fillna(0.0) >= COPY_MIN_COPYABLE_ROI)
)
copy_wallets = set(wallet_vol.loc[copy_mask, 'wallet'])
print(f"Copy universe: {len(copy_wallets)} wallets")

if TEST_MODE and len(copy_wallets) > MAX_CANDIDATE_WALLETS:
    copy_wallets = set(
        wallet_vol.loc[copy_mask].sort_values('buy_roi', ascending=False)
        .head(MAX_CANDIDATE_WALLETS)['wallet']
    )
    print(f"  (test mode: {len(copy_wallets)} wallets)")

# BUY trades by copy-universe wallets
candidate_mask = df_full['wallet'].isin(copy_wallets) & (df_full['side'] == 'BUY')
candidate_trades = df_full[candidate_mask].copy()
print(f"Candidate BUY trades: {len(candidate_trades):,}")

if TEST_MODE and len(candidate_trades) > MAX_CANDIDATE_TRADES:
    candidate_trades = candidate_trades.sample(MAX_CANDIDATE_TRADES, random_state=42)
    print(f"  (test mode: sampled {len(candidate_trades)})")


Copy universe: 58 wallets
Candidate BUY trades: 140,802


## Signal 1: Bad Leader Proximity

**Hypothesis:** Trades on the same (condition_id, outcome) recently after a
"bad leader" (profitable but uncopyable wallet) tend to underperform.

Already computed via `merge_asof` above. Column: `bad_leader_wallet`.

In [778]:

df_train, df_val, df_test = split_data(df_full, method='chronological')

c_train = df_train[df_train['wallet'].isin(copy_wallets) & (df_train['side'] == 'BUY')].copy()
c_val = df_val[df_val['wallet'].isin(copy_wallets) & (df_val['side'] == 'BUY')].copy()
c_test = df_test[df_test['wallet'].isin(copy_wallets) & (df_test['side'] == 'BUY')].copy()

print(f"  Train: {len(c_train):,}  Val: {len(c_val):,}  Test: {len(c_test):,}")
# Signal 1: bad_leader_buy (binary)
# Already computed in earlier cells (bad_leader_wallet column on df_full)
c_train['sig_bad_leader'] = c_train['bad_leader_wallet'].notna().astype(float)
c_val['sig_bad_leader'] = c_val['bad_leader_wallet'].notna().astype(float)
c_test['sig_bad_leader'] = c_test['bad_leader_wallet'].notna().astype(float)

print("Signal 1: bad_leader_buy (binary)")
for label, df_c in [("Train", c_train), ("Val", c_val), ("Test", c_test)]:
    rate = df_c['sig_bad_leader'].mean()
    ic_v = compute_event_ic(df_c['sig_bad_leader'], df_c['copyable_roi'])
    print(f"  {label}: firing_rate={rate:.4f}  IC={ic_v:.4f}")


Chronological split: train <= 2026-05-21T00:00:00Z, val <= 2026-06-23T00:00:00Z, test > 2026-06-23T00:00:00Z
Method: chronological  |  Unique end dates: 112  (train=44, val=33, test=35)

  Train:  4,337,961 trades  (15,923 markets)
  Val:    5,211,194 trades  (20,297 markets)
  Test:   4,701,448 trades  (20,655 markets)
  Total: 14,250,603 trades  (56,875 markets)
  Train: 30,448  Val: 61,289  Test: 49,065
Signal 1: bad_leader_buy (binary)
  Train: firing_rate=0.0160  IC=0.0105
  Val: firing_rate=0.0108  IC=0.0105
  Test: firing_rate=0.0076  IC=0.0088


## Signal 2: Quality Wallet Proximity

**Hypothesis:** When high-quality wallets buy a token, copying within a
window yields positive PnL. Quality wallets have high buy_roi, diversification,
and stable returns.

In [779]:

# Signal 2: quality_wallet_proximity

# Define quality wallets from training-period metrics
qw_mask = (
    (wallet_vol['buy_roi'] >= QW_MIN_BUY_ROI)
    & (wallet_vol['num_buckets'] >= QW_MIN_BUCKETS)
    & (wallet_vol['trade_count'] >= QW_MIN_TRADE_COUNT)
    & (wallet_vol['max_drawdown_to_pnl'].fillna(1.0) <= QW_MAX_DD_TO_PNL)
    & (wallet_vol['copyable_roi'].fillna(0.0) >= QW_MIN_COPYABLE_ROI)
)
quality_wallets = set(wallet_vol.loc[qw_mask, 'wallet'])
print(f"Quality wallets: {len(quality_wallets)}")

# merge_asof: for each trade, find nearest quality-wallet buy on same
# (condition_id, outcome) in last QUALITY_WALLET_WINDOW minutes
# NOTE: old candidate slices (c_train/c_val/c_test) do NOT have this column yet;
# we will re-slice them from df_full in the next cell.
if 'qw_wallet' not in df_full.columns:
    qw_buys = df_full[
        df_full['wallet'].isin(quality_wallets) & (df_full['side'] == 'BUY')
    ].copy()
    qw_buys = qw_buys.rename(columns={
        'dt': 'dt_qw', 'wallet': 'qw_wallet', 'usdc_amount': 'qw_usdc'
    })[['dt_qw', 'qw_wallet', 'qw_usdc', 'condition_id', 'outcome']].sort_values('dt_qw')
    print(f"  Quality wallet BUY trades: {len(qw_buys):,}")

    df_full = pd.merge_asof(
        df_full.sort_values('dt'),
        qw_buys,
        left_on='dt', right_on='dt_qw',
        by=['condition_id', 'outcome'],
        direction='backward',
        tolerance=pd.Timedelta(minutes=QUALITY_WALLET_WINDOW),
        allow_exact_matches=False,
        suffixes=('', '_qw'),
    )
    print("  Signal 2 column 'qw_wallet' added to df_full")
else:
    print("  Signal 2 already computed, skipping merge_asof")

# Window stats: quality-wallet trading activity per (condition_id, outcome, time window)
qw_buys_for_stats = df_full[
    df_full['wallet'].isin(quality_wallets) & (df_full['side'] == 'BUY')
].copy()
qw_buys_for_stats['dt_window'] = qw_buys_for_stats['dt'].dt.floor('15min')

qw_roi_map = wallet_vol.set_index('wallet')['copyable_roi'].to_dict()
qw_buys_for_stats['wallet_roi'] = qw_buys_for_stats['wallet'].map(qw_roi_map)

qw_window_stats = qw_buys_for_stats.groupby(
    ['condition_id', 'outcome', 'dt_window'], sort=False, observed=True
).agg(
    qw_trade_count=('wallet', 'size'),
    qw_unique_wallets=('wallet', 'nunique'),
    qw_total_volume=('usdc_amount', 'sum'),
    qw_roi_sum=('wallet_roi', 'sum'),
).reset_index()

df_full['dt_window'] = df_full['dt'].dt.floor('15min')
df_full = df_full.merge(
    qw_window_stats,
    on=['condition_id', 'outcome', 'dt_window'],
    how='left',
)

qw_window_stats['opposite_outcome'] = qw_window_stats['outcome'].map({'No': 'Yes', 'Yes': 'No'})
qw_window_stats.rename(columns={'qw_trade_count': 'qw_trade_count_opposite',
                                 'qw_unique_wallets': 'qw_unique_wallets_opposite', 
                                 'qw_total_volume': 'qw_total_volume_opposite', 
                                 'qw_roi_sum': 'qw_roi_sum_opposite'}, inplace=True)

df_full = df_full.merge(
    qw_window_stats[['condition_id', 'opposite_outcome', 'dt_window', 'qw_trade_count_opposite', 'qw_unique_wallets_opposite', 'qw_total_volume_opposite', 'qw_roi_sum_opposite']],
    left_on=['condition_id', 'outcome', 'dt_window'],
    right_on=['condition_id', 'opposite_outcome', 'dt_window'],
    how='left',
    suffixes=('', '_opposite'),
)
print(f"  Window stats added: {len(qw_window_stats)} (condition_id, outcome, 15min) windows")
print("  New columns: qw_trade_count, qw_unique_wallets, qw_total_volume, qw_roi_sum, qw_trade_count_opposite, qw_unique_wallets_opposite, qw_total_volume_opposite, qw_roi_sum_opposite")

# Candidate sets will be refreshed from df_full in the next cell,
# at which point they will have qw_wallet + qw_usdc + window stats columns.
print("  (candidate trades will be re-sliced next)")


Quality wallets: 27
  Signal 2 already computed, skipping merge_asof


MergeError: Passing 'suffixes' which cause duplicate columns {'qw_total_volume_x', 'qw_roi_sum_x', 'qw_unique_wallets_x', 'qw_trade_count_x'} is not allowed.

## Refresh candidate trades

Re-slice candidate trades from the fully-annotated `df_full` (now includes quality_wallet signal + window stats).

In [ ]:
# # Re-slice candidate trades after signal columns added to df_full
c_train, c_val, c_test = split_data(df_full[df_full['wallet'].isin(copy_wallets) & (df_full['side'] == 'BUY')], method='chronological')

Chronological split: train <= 2026-05-31T00:00:00Z, val <= 2026-06-27T00:00:00Z, test > 2026-06-27T00:00:00Z
Method: chronological  |  Unique end dates: 92  (train=36, val=27, test=29)

  Train:     46,338 trades  (7,717 markets)
  Val:       52,217 trades  (10,200 markets)
  Test:      42,247 trades  (8,990 markets)
  Total:    140,802 trades  (26,907 markets)


## Assign signal columns

Derive signal columns from `df_full` columns on refreshed candidate sets.

In [780]:
df_train[df_train['bad_leader'].notna()][['dt', 'wallet', 'condition_id', 'outcome', 'bad_leader']].head(10)

,dt,wallet,condition_id,outcome,bad_leader
19029,2026-04-28 00:31:24+00:00,0xb06a0eae498750ed0acac7e1f759f741c56e52f5,0x06ada32ccc6deb035317b855a18397eaefc85baf4838...,Yes,1.0
19663,2026-04-28 00:36:52+00:00,0x7ce336595687024c9ebf620f2dcb3b71269b1689,0x06ada32ccc6deb035317b855a18397eaefc85baf4838...,Yes,1.0
22369,2026-04-28 01:11:58+00:00,0x8cc1262cf53b83047926e09af8e5ab3bf537b9f8,0x124d23be9b7ea2193b991def2f3739082facdf29bcc8...,Yes,1.0
22865,2026-04-28 01:19:52+00:00,0xf229d2b21a9183904c56cddeecbe6ab7c4ca3058,0x0815986e0d935d7557e7cbd632e55a66e4a1fed13884...,Yes,1.0
22924,2026-04-28 01:20:18+00:00,0x6d58bd71c53b4c292a72b97a03a32e41a6e617a9,0x0815986e0d935d7557e7cbd632e55a66e4a1fed13884...,Yes,1.0
23072,2026-04-28 01:22:20+00:00,0x26123cbf0f4820f7e70408a8c054ba7615c05289,0x152a2810cb144f564293253c2108400b9ef2568332e2...,Yes,1.0
23078,2026-04-28 01:22:24+00:00,0xf56ccbb9d8158ae1dda67207d1f76acf2f81ad84,0x152a2810cb144f564293253c2108400b9ef2568332e2...,Yes,1.0
23095,2026-04-28 01:22:52+00:00,0x26123cbf0f4820f7e70408a8c054ba7615c05289,0x152a2810cb144f564293253c2108400b9ef2568332e2...,Yes,1.0
23096,2026-04-28 01:22:52+00:00,0x1c06e4a2c157c07872f9139a3f23806ed0d562e3,0x152a2810cb144f564293253c2108400b9ef2568332e2...,Yes,1.0
23097,2026-04-28 01:22:52+00:00,0x6d58bd71c53b4c292a72b97a03a32e41a6e617a9,0x152a2810cb144f564293253c2108400b9ef2568332e2...,Yes,1.0


In [781]:
# Assign signal columns on refreshed candidate sets
# (df_full already has bad_leader_wallet, qw_wallet, and window stats columns)
for df_c in [c_train, c_val, c_test]:
    # Signal 1: bad leader proximity
    df_c['sig_bad_leader'] = df_c['bad_leader_wallet'].notna().astype(float)
    # Signal 2: quality wallet proximity
    df_c['sig_qw_any'] = df_c['qw_wallet'].notna().astype(float)
    df_c['sig_qw_volume'] = df_c['qw_usdc'].fillna(0.0)
    # Signal 3: quality wallet window stats
    # df_c['sig_qw_consensus'] = df_c['qw_unique_wallets'].fillna(0.0)
        # df_c['sig_qw_freq'] = df_c['qw_trade_count'].fillna(0.0)
        # df_c['sig_qw_reputation'] = df_c['qw_roi_sum'].fillna(0.0)
    # Signal 4: quality wallet opposite window stats
    df_c['sig_qw_consensus_opposite'] = df_c['qw_unique_wallets_opposite'].fillna(0.0)
    df_c['sig_qw_freq_opposite'] = df_c['qw_trade_count_opposite'].fillna(0.0)
    df_c['sig_qw_reputation_opposite'] = df_c['qw_roi_sum_opposite'].fillna(0.0)

    for sig in selected_filter_names:
        df_c[sig] = df_c[sig].notna().astype(float)
    # combi
    # df_c['sig_qw_disagreement'] = df_c[['sig_qw_freq', 'sig_qw_freq_opposite']].min(axis=1)

# Print IC for each signal
for sig in ['sig_bad_leader', 'sig_qw_any', 'sig_qw_volume',
            # 'sig_qw_consensus', 'sig_qw_freq', 'sig_qw_reputation',
            'sig_qw_consensus_opposite', 'sig_qw_freq_opposite', 'sig_qw_reputation_opposite',
            # 'sig_qw_disagreement'
            ] + selected_filter_names:
    print(f"{sig}:")
    for label, df_c in [("Train", c_train), ("Val", c_val), ("Test", c_test)]:
        rate = df_c[sig].mean() if df_c[sig].dtype.kind in 'bif' else df_c[sig].notna().mean()
        ic_v = compute_event_ic(df_c[sig], df_c['copyable_roi'])
        print(f"  {label}: firing_rate={rate:.4f}  IC={ic_v:.4f}")
    print()
for sig in ['sig_vwap_csrank', 'sig_vwap_signed']:
    if sig in c_train.columns:
        print(f"{sig}:")
        for label, df_c in [("Train", c_train), ("Val", c_val), ("Test", c_test)]:
            ic_cr = compute_event_ic(df_c[sig], df_c['copyable_roi'])
            ic_pnl = compute_event_ic(df_c[sig], df_c['pnl'])
            print(f"  {label}: IC(copyable_roi)={ic_cr:.4f}  IC(pnl)={ic_pnl:.4f}")
        print()




sig_bad_leader:
  Train: firing_rate=0.0160  IC=0.0105
  Val: firing_rate=0.0108  IC=0.0105
  Test: firing_rate=0.0076  IC=0.0088

sig_qw_any:
  Train: firing_rate=0.2506  IC=-0.0060
  Val: firing_rate=0.1078  IC=0.0015
  Test: firing_rate=0.0900  IC=-0.0005

sig_qw_volume:
  Train: firing_rate=4.2130  IC=-0.0079
  Val: firing_rate=2.5388  IC=0.0020
  Test: firing_rate=3.4774  IC=-0.0026

sig_qw_consensus_opposite:
  Train: firing_rate=0.0849  IC=0.0110
  Val: firing_rate=0.0776  IC=-0.0103
  Test: firing_rate=0.0664  IC=0.0061

sig_qw_freq_opposite:
  Train: firing_rate=0.2128  IC=0.0105
  Val: firing_rate=0.1821  IC=-0.0103
  Test: firing_rate=0.1594  IC=0.0051

sig_qw_reputation_opposite:
  Train: firing_rate=0.0037  IC=0.0142
  Val: firing_rate=0.0035  IC=-0.0047
  Test: firing_rate=0.0029  IC=0.0016

bad_leader:
  Train: firing_rate=0.0160  IC=0.0105
  Val: firing_rate=0.0108  IC=0.0105
  Test: firing_rate=0.0076  IC=0.0088

bad_leader_opposite:
  Train: firing_rate=0.0128  IC=-0.

## Signal 3: VWAP Deviation (simplified)

**Hypothesis:** Buying below the trailing VWAP of quality buyers is a better entry.
VWAP deviation = (price / vwap_15m) - 1.

For TEST_MODE: bucketed 5-min VWAP approximation.

In [ ]:

# Signal 3: VWAP deviation (simplified for TEST_MODE)

VWAP_BUCKET_MINUTES = 5

if TEST_MODE:
    print("TEST_MODE: bucketed VWAP approximation")

    def floor_dt(s, freq=f"{VWAP_BUCKET_MINUTES}min"):
        return s.dt.floor(freq)

    # All BUY trades by copy-universe wallets
    copy_buys = df_full[
        df_full['wallet'].isin(copy_wallets) & (df_full['side'] == 'BUY')
    ].copy()
    copy_buys['dt_bucket'] = floor_dt(copy_buys['dt'])

    # Per (condition_id, outcome, bucket): VWAP
    bucket_vwap = copy_buys.groupby(
        ['condition_id', 'outcome', 'dt_bucket'], sort=False
    ).apply(
        lambda g: pd.Series({
            'vwap': (g['price'] * g['quantity']).sum() / g['quantity'].sum(),
            'vwap_vol': g['usdc_amount'].sum(),
        }), include_groups=False
    ).reset_index()

    # Shift VWAP one bucket forward to avoid look-ahead
    bucket_vwap['dt_bucket_prev'] = bucket_vwap['dt_bucket'] - pd.Timedelta(minutes=VWAP_BUCKET_MINUTES)

    # For each candidate trade, merge on previous bucket
    _vwap_dfs = []
    for name, df_c in [('train', c_train), ('val', c_val), ('test', c_test)]:
        df_c['dt_bucket'] = floor_dt(df_c['dt'])
        df_c = df_c.merge(
            bucket_vwap,
            left_on=['condition_id', 'outcome', 'dt_bucket'],
            right_on=['condition_id', 'outcome', 'dt_bucket_prev'],
            how='left',
            suffixes=('', '_vwap'),
        )
        df_c['sig_vwap_dev'] = np.where(
            df_c['vwap'].notna() & (df_c['vwap'] > 0),
            (df_c['price'] / df_c['vwap']) - 1.0,
            np.nan,
        )
        # Signed: negative z-score = buying below VWAP = good entry
        df_c['sig_vwap_signed'] = np.where(
            df_c['sig_vwap_dev'].notna(),
            -df_c['sig_vwap_dev'],
            np.nan,
        )
        # Also keep magnitude for comparison
        df_c['sig_vwap_strength'] = df_c['sig_vwap_dev'].abs().fillna(0.0)
        _vwap_dfs.append((name, df_c))
    for name, df_c in _vwap_dfs:
        if name == 'train': c_train = df_c
        elif name == 'val': c_val = df_c
        elif name == 'test': c_test = df_c

else:
    print("Non-TEST_MODE: vectorized bucketed VWAP (previous bucket)")

    copy_buys = df_full[
        df_full['wallet'].isin(copy_wallets) & (df_full['side'] == 'BUY')
    ].copy()
    copy_buys['dt_bucket'] = copy_buys['dt'].dt.floor(f"{VWAP_BUCKET_MINUTES}min")

    copy_buys['price_vol'] = copy_buys['price'] * copy_buys['quantity']
    bucket_vwap = copy_buys.groupby(
        ['condition_id', 'outcome', 'dt_bucket'], sort=False, observed=True
    ).agg(
        vwap_price=('price_vol', 'sum'),
        total_qty=('quantity', 'sum'),
        vwap_vol=('usdc_amount', 'sum'),
    ).reset_index()
    bucket_vwap['vwap'] = bucket_vwap['vwap_price'] / bucket_vwap['total_qty']

    bucket_vwap['dt_bucket_prev'] = bucket_vwap['dt_bucket'] - pd.Timedelta(minutes=VWAP_BUCKET_MINUTES)

    _vwap_dfs = []
    for name, df_c in [('train', c_train), ('val', c_val), ('test', c_test)]:
        df_c['dt_bucket'] = df_c['dt'].dt.floor(f"{VWAP_BUCKET_MINUTES}min")
        df_c = df_c.merge(
            bucket_vwap[['condition_id', 'outcome', 'dt_bucket_prev', 'vwap', 'vwap_vol']],
            left_on=['condition_id', 'outcome', 'dt_bucket'],
            right_on=['condition_id', 'outcome', 'dt_bucket_prev'],
            how='left',
            suffixes=('', '_vwap'),
        )
        df_c['sig_vwap_dev'] = np.where(
            df_c['vwap'].notna() & (df_c['vwap'] > 0),
            (df_c['price'] / df_c['vwap']) - 1.0,
            np.nan,
        )
        df_c['sig_vwap_signed'] = np.where(
            df_c['sig_vwap_dev'].notna(),
            -df_c['sig_vwap_dev'],
            np.nan,
        )
        df_c['sig_vwap_strength'] = df_c['sig_vwap_dev'].abs().fillna(0.0)
        _vwap_dfs.append((name, df_c))
    for name, df_c in _vwap_dfs:
        if name == 'train': c_train = df_c
        elif name == 'val': c_val = df_c
        elif name == 'test': c_test = df_c

# CS-rank VWAP signed deviation for comparability across days
for df_c in [c_train, c_val, c_test]:
    df_c['sig_vwap_csrank'] = cs_rank(df_c['sig_vwap_signed'].fillna(0.0), df_c['dt'].dt.date)

print()
for label, df_c in [("Train", c_train), ("Val", c_val), ("Test", c_test)]:
    cov = df_c['sig_vwap_dev'].notna().mean()
    ic_signed = compute_event_ic(df_c['sig_vwap_signed'], df_c['copyable_roi'])
    ic_strength = compute_event_ic(df_c['sig_vwap_strength'], df_c['copyable_roi'])
    ic_csrank = compute_event_ic(df_c['sig_vwap_csrank'], df_c['copyable_roi'])
    print(f"  {label}: coverage={cov:.3f}  IC(signed)={ic_signed:.4f}  IC(strength)={ic_strength:.4f}  IC(csrank)={ic_csrank:.4f}")


Non-TEST_MODE: vectorized bucketed VWAP (previous bucket)

  Train: coverage=0.237  IC(signed)=0.0031  IC(strength)=0.0113  IC(csrank)=-0.0037
  Val: coverage=0.142  IC(signed)=0.0005  IC(strength)=-0.0041  IC(csrank)=-0.0097
  Test: coverage=0.145  IC(signed)=0.0003  IC(strength)=0.0029  IC(csrank)=0.0044


## Signal Quality Report

Compute IC, IR, hit rate, and bootstrap confidence intervals on validation.

In [ ]:

# Signal quality report on validation set
signal_cols = ['sig_bad_leader', 'sig_qw_any',
               'sig_qw_consensus', 'sig_qw_freq', 'sig_qw_reputation',
               'sig_vwap_signed', 'sig_vwap_strength', 'sig_vwap_csrank']
active_cols = [c for c in signal_cols if c in c_val.columns and c_val[c].notna().sum() > 10]

quality_report = signal_quality_report(
    c_val, active_cols,
    roi_col='copyable_roi', dt_col='dt',
    ir_freq='D', n_bootstrap=5_000,
)

print("Signal quality report (validation set):")
display(quality_report.round(4))


Signal quality report: 5 signals, 52217 events
Signal quality report (validation set):


,signal,IC,IR,hit_rate,bootstrap_mean_ic,bootstrap_ci_lo,bootstrap_ci_hi,n_events
4,sig_vwap_csrank,-0.0097,0.0910,0.5084,-1,-1,-1,52217
3,sig_vwap_strength,-0.0041,-0.2477,0.0616,-1,-1,-1,52217
1,sig_qw_any,0.0015,0.1405,0.0530,-1,-1,-1,52217
0,sig_bad_leader,0.0009,0.1781,0.0090,-1,-1,-1,52217
2,sig_vwap_signed,0.0005,-0.2718,0.4986,-1,-1,-1,7405


## Signal Overlap Analysis

How redundant are the signals? Do they fire on the same events?

In [ ]:

# Overlap analysis
active_cols = [c for c in signal_cols if c in c_val.columns and c_val[c].notna().sum() > 10]
n_sig = len(active_cols)

if n_sig < 2:
    print("Need at least 2 active signals for overlap analysis")
else:
    # 1. Coincidence rate
    print("1. Coincidence rate (P(both fire | either fires)):")
    coin_mat = np.full((n_sig, n_sig), np.nan)
    for i, s1 in enumerate(active_cols):
        for j, s2 in enumerate(active_cols):
            coin_mat[i, j] = 1.0 if i == j else coincidence_rate(c_val[s1], c_val[s2])
    coin_df = pd.DataFrame(coin_mat, index=active_cols, columns=active_cols)
    display(coin_df.round(3))

    # 2. IC correlation
    print("\n2. IC correlation (signal value correlation):")
    ic_corr = ic_correlation_matrix(c_val, active_cols)
    display(ic_corr.round(3))

    # 3. Conditional IC
    print("\n3. Conditional IC (unique contribution):")
    for s in active_cols:
        other = [c for c in active_cols if c != s]
        neutral = np.ones(len(c_val), dtype=bool)
        for o in other:
            neutral &= (c_val[o].abs() < 0.01) | c_val[o].isna()
        if neutral.sum() < 20:
            continue
        ic_cond = compute_event_ic(c_val.loc[neutral, s], c_val.loc[neutral, 'copyable_roi'])
        ic_full = compute_event_ic(c_val[s], c_val['copyable_roi'])
        print(f"    {s:25s}: full_IC={ic_full:.4f}  conditional_IC={ic_cond:.4f}  "
              f"(n={neutral.sum()})")


1. Coincidence rate (P(both fire | either fires)):


,sig_bad_leader,sig_qw_any,sig_vwap_signed,sig_vwap_strength,sig_vwap_csrank
sig_bad_leader,1.000,0.084,0.022,0.022,0.011
sig_qw_any,0.084,1.000,0.099,0.099,0.094
sig_vwap_signed,0.022,0.099,1.000,1.000,0.107
sig_vwap_strength,0.022,0.099,1.000,1.000,0.107
sig_vwap_csrank,0.011,0.094,0.107,0.107,1.000



2. IC correlation (signal value correlation):


,sig_bad_leader,sig_qw_any,sig_vwap_signed,sig_vwap_strength,sig_vwap_csrank
sig_bad_leader,1.000,0.003,0.002,-0.000,0.011
sig_qw_any,0.003,1.000,-0.023,0.004,-0.004
sig_vwap_signed,0.002,-0.023,1.000,-0.003,0.003
sig_vwap_strength,-0.000,0.004,-0.003,1.000,0.003
sig_vwap_csrank,0.011,-0.004,0.003,0.003,1.000



3. Conditional IC (unique contribution):
    sig_bad_leader           : full_IC=0.0009  conditional_IC=0.0096  (n=10260)
    sig_qw_any               : full_IC=0.0015  conditional_IC=0.0104  (n=11113)
    sig_vwap_signed          : full_IC=0.0005  conditional_IC=-0.0568  (n=10239)
    sig_vwap_strength        : full_IC=-0.0041  conditional_IC=0.0106  (n=10239)
    sig_vwap_csrank          : full_IC=-0.0097  conditional_IC=0.0005  (n=43802)


## Signal Combination

Combine signals into a composite score:
1. **Equal weight**: w_i = 1/n
2. **IC weight**: w_i = IC_i / sum|IC_j|
3. **Shrinkage Markowitz**: (1-lambda)*inv(Sigma)*IC + lambda/n

In [ ]:

# Signal combination methods
active_cols = [c for c in signal_cols if c in c_val.columns and c_val[c].notna().sum() > 10]

if not active_cols:
    print("No active signals found")
else:
    print(f"Combining {len(active_cols)} signals: {active_cols}")

    # 1. Equal weight
    w_equal = pd.Series(1.0 / len(active_cols), index=active_cols)

    # 2. IC weight
    ic_vals = {c: compute_event_ic(c_val[c], c_val['copyable_roi']) or 0.0
               for c in active_cols}
    ic_sum = sum(abs(v) for v in ic_vals.values())
    w_ic = pd.Series({c: ic_vals[c] / ic_sum if ic_sum > 0 else 1.0/len(active_cols)
                       for c in active_cols})

    # 3. Shrinkage Markowitz
    w_shrink = compute_optimal_weights(c_val, active_cols, 'copyable_roi', shrinkage=0.5)

    schemes = {
        'equal': w_equal,
        'ic_weighted': w_ic,
        'shrinkage_markowitz': w_shrink,
    }

    for name, w in schemes.items():
        print(f"\n  {name}:")
        for c, wt in w.items():
            print(f"    {c:25s} = {wt:.4f}")

    # Apply composite scores
    for name, w in schemes.items():
        for df_c in [c_train, c_val, c_test]:
            df_c[f'composite_{name}'] = apply_composite_score(df_c, active_cols, w)

    # Compare on validation
    comp_cols = [f'composite_{k}' for k in schemes]
    comp_results = []
    for cc in comp_cols:
        ic_c = compute_event_ic(c_val[cc], c_val['copyable_roi'])
        ir_c = compute_event_ir(c_val[cc], c_val['copyable_roi'], c_val['dt'], freq='D')
        comp_results.append({'composite': cc, 'IC': ic_c, 'IR': ir_c})
    comp_df = pd.DataFrame(comp_results)
    print("\n\nComposite signal quality (validation):")
    display(comp_df.round(4))


Combining 5 signals: ['sig_bad_leader', 'sig_qw_any', 'sig_vwap_signed', 'sig_vwap_strength', 'sig_vwap_csrank']

  equal:
    sig_bad_leader            = 0.2000
    sig_qw_any                = 0.2000
    sig_vwap_signed           = 0.2000
    sig_vwap_strength         = 0.2000
    sig_vwap_csrank           = 0.2000

  ic_weighted:
    sig_bad_leader            = 0.0565
    sig_qw_any                = 0.0879
    sig_vwap_signed           = 0.0271
    sig_vwap_strength         = -0.2437
    sig_vwap_csrank           = -0.5848

  shrinkage_markowitz:
    sig_bad_leader            = 0.0598
    sig_qw_any                = 0.0929
    sig_vwap_signed           = -0.0907
    sig_vwap_strength         = -0.1384
    sig_vwap_csrank           = -0.6183


Composite signal quality (validation):


,composite,IC,IR
0,composite_equal,-0.0062,-0.0626
1,composite_ic_weighted,-0.0011,0.0058
2,composite_shrinkage_markowitz,0.0011,0.0160


## Strategy Evaluation

When composite_score >= threshold, copy the BUY trade. Grid-search threshold on validation.

In [ ]:

# Strategy evaluation: when composite_score >= threshold, copy the trade
# Use Markowitz composite if available, fall back to IC-weighted, then equal

best_composite = 'composite_shrinkage_markowitz'
if best_composite not in c_val.columns or c_val[best_composite].notna().sum() < 10:
    best_composite = 'composite_ic_weighted'
if best_composite not in c_val.columns or c_val[best_composite].notna().sum() < 10:
    best_composite = 'composite_equal'
print(f"Using: {best_composite}")


def evaluate_strategy(df, score_col, threshold):
    fired = df[df[score_col] >= threshold].copy()
    if fired.empty:
        return {
            'threshold': threshold, 'trades': 0,
            'copyable_pnl': 0.0, 'copyable_roi': 0.0,
            'total_pnl': 0.0, 'notional': 0.0,
            'copyable_notional': 0.0, 'firing_rate': 0.0,
        }
    cnot = fired['copyable_notional'].sum()
    return {
        'threshold': threshold,
        'trades': len(fired),
        'copyable_pnl': float(fired['copyable_pnl'].sum()),
        'copyable_roi': float(fired['copyable_pnl'].sum() / cnot) if cnot > 0 else 0.0,
        'total_pnl': float(fired['pnl'].sum()),
        'notional': float(fired['notional'].sum()),
        'copyable_notional': float(cnot),
        'firing_rate': len(fired) / len(df),
    }


# Grid search on validation
thresholds = np.arange(0.0, 1.05, 0.05)
val_results = [evaluate_strategy(c_val, best_composite, t) for t in thresholds]
val_df = pd.DataFrame(val_results)
val_df['pnl_per_trade'] = val_df['copyable_pnl'] / val_df['trades'].clip(lower=1)

print("\nGrid search (validation): top 10 by copyable_pnl")
display(val_df.sort_values('copyable_pnl', ascending=False).head(10).round(2))

# Pick best threshold (max copyable_pnl, min 20 trades)
candidates = val_df[val_df['trades'] >= 20]
if not candidates.empty:
    best_row = candidates.sort_values('copyable_pnl', ascending=False).iloc[0]
else:
    best_row = val_df.sort_values('copyable_pnl', ascending=False).iloc[0]
best_threshold = best_row['threshold']
print(f"\nBest threshold: {best_threshold:.2f}  "
      f"(copyable_pnl=${best_row['copyable_pnl']:,.0f}, "
      f"{best_row['trades']} trades)")


Using: composite_shrinkage_markowitz

Grid search (validation): top 10 by copyable_pnl


,threshold,trades,copyable_pnl,copyable_roi,total_pnl,notional,copyable_notional,firing_rate,pnl_per_trade
2,0.10,5144,1147.06,0.03,5962.79,124396.03,34718.95,0.10,0.22
1,0.05,6636,1140.46,0.03,8018.26,164819.49,45493.70,0.13,0.17
3,0.15,2619,833.92,0.07,2706.75,54796.08,12402.00,0.05,0.32
0,0.00,45474,514.55,0.00,18358.28,840348.94,186777.66,0.87,0.01
10,0.50,2309,473.31,0.06,1493.52,39566.36,8078.16,0.04,0.20
8,0.40,2316,466.89,0.06,1485.58,39579.66,8088.58,0.04,0.20
6,0.30,2318,466.46,0.06,1485.12,39580.12,8089.00,0.04,0.20
7,0.35,2318,466.46,0.06,1485.12,39580.12,8089.00,0.04,0.20
9,0.45,2314,465.92,0.06,1484.60,39576.63,8085.55,0.04,0.20
5,0.25,2319,453.55,0.06,1472.20,39593.03,8101.92,0.04,0.20



Best threshold: 0.10  (copyable_pnl=$1,147, 5144.0 trades)


In [ ]:

# Test set evaluation
test_result = evaluate_strategy(c_test, best_composite, best_threshold)
all_result = evaluate_strategy(c_test, best_composite, -np.inf)

print("Test set evaluation:")
print(f"  Threshold: {best_threshold:.2f}")
print(f"  Trades fired: {test_result['trades']:,} / {all_result['trades']:,} "
      f"({test_result['firing_rate']:.1%})")
print(f"  Copyable PnL: ${test_result['copyable_pnl']:,.0f}")
print(f"  Copyable ROI: {test_result['copyable_roi']:.4f}")
print(f"  Total PnL: ${test_result['total_pnl']:,.0f}")
print(f"  PnL per trade: ${test_result['copyable_pnl'] / max(test_result['trades'], 1):.2f}")
print()
print("vs. copying all candidate trades:")
print(f"  Copyable PnL (all): ${all_result['copyable_pnl']:,.0f}")
print(f"  Copyable ROI (all): {all_result['copyable_roi']:.4f}")

# Summary across splits
print("\n\n=== Strategy Summary ===")
for label, df_i, th in [('Train', c_train, best_threshold),
                          ('Val', c_val, best_threshold),
                          ('Test', c_test, best_threshold)]:
    r = evaluate_strategy(df_i, best_composite, th)
    ra = evaluate_strategy(df_i, best_composite, -np.inf)
    print(f"  {label:6s}: threshold={th:.2f}  "
          f"trades={r['trades']:>5,}/{len(df_i):>6,}  "
          f"cpnl=${r['copyable_pnl']:>8,.0f}  "
          f"croi={r['copyable_roi']:.4f}  "
          f"(all: cpnl=${ra['copyable_pnl']:>8,.0f}  croi={ra['copyable_roi']:.4f})")


Test set evaluation:
  Threshold: 0.10
  Trades fired: 4,030 / 42,247 (9.5%)
  Copyable PnL: $238
  Copyable ROI: 0.0074
  Total PnL: $3,521
  PnL per trade: $0.06

vs. copying all candidate trades:
  Copyable PnL (all): $9,605
  Copyable ROI (all): 0.0468


=== Strategy Summary ===
  Train : threshold=0.10  trades=8,700/46,338  cpnl=$   7,352  croi=0.1225  (all: cpnl=$  29,562  croi=0.1076)
  Val   : threshold=0.10  trades=5,144/52,217  cpnl=$   1,147  croi=0.0330  (all: cpnl=$   2,070  croi=0.0095)
  Test  : threshold=0.10  trades=4,030/42,247  cpnl=$     238  croi=0.0074  (all: cpnl=$   9,605  croi=0.0468)


In [ ]:

# Leave-one-out signal contribution (validation)
active_cols = [c for c in signal_cols if c in c_val.columns and c_val[c].notna().sum() > 10]

if len(active_cols) >= 2:
    print("Signal contribution (leave-one-out on validation):")
    full_ic = compute_event_ic(c_val[best_composite], c_val['copyable_roi'])

    loo_results = []
    for leave_out in active_cols:
        remaining = [c for c in active_cols if c != leave_out]
        if not remaining:
            continue
        w = compute_optimal_weights(c_val, remaining, 'copyable_roi', shrinkage=0.5)
        c_val[f'composite_loo_{leave_out}'] = apply_composite_score(c_val, remaining, w)
        ic_loo = compute_event_ic(c_val[f'composite_loo_{leave_out}'], c_val['copyable_roi'])
        loo_results.append({'left_out': leave_out, 'IC': ic_loo, 'IC_drop': full_ic - ic_loo})

    loo_df = pd.DataFrame(loo_results).sort_values('IC_drop', ascending=False)
    print(f"  Full composite IC: {full_ic:.4f}")
    display(loo_df.round(4))


Signal contribution (leave-one-out on validation):
  Full composite IC: 0.0011


,left_out,IC,IC_drop
2,sig_vwap_signed,-0.0025,0.0036
0,sig_bad_leader,0.0009,0.0002
1,sig_qw_any,0.0088,-0.0077
4,sig_vwap_csrank,0.0090,-0.0079
3,sig_vwap_strength,0.0109,-0.0098


## Save Results

In [ ]:

# Persist results
import json
from datetime import datetime, timezone
from pathlib import Path

# Signal ICs on TRAIN only (no test leakage)
signal_ics = {}
for col in active_cols:
    signal_ics[col] = {
        "IC": compute_event_ic(c_train[col], c_train['copyable_roi']),
        "IR": compute_event_ir(c_train[col], c_train['copyable_roi'], c_train['dt'], freq='D'),
        "hit_rate": hit_rate(c_train[col], c_train['copyable_roi']),
    }

output = {
    "stage": 1,
    "type": "experimental_signal_framework",
    "metadata": {
        "run_timestamp": datetime.now(timezone.utc).isoformat(),
        "TEST_MODE": TEST_MODE,
        "n_copy_wallets": len(copy_wallets),
        "n_quality_wallets": len(quality_wallets),
        "n_candidate_trades": len(candidate_trades),
        "best_threshold": float(best_threshold),
        "best_composite": best_composite,
        "signal_windows_min": {
            "bad_leader": BAD_LEADER_WINDOW,
            "quality_wallet": QUALITY_WALLET_WINDOW,
            "vwap": VWAP_WINDOW,
        },
    },
    "signals": {
        col: vals for col, vals in signal_ics.items()
    },
    "weights": {
        "equal": {k: float(v) for k, v in w_equal.items()},
        "ic_weighted": {k: float(v) for k, v in w_ic.items()},
        "shrinkage_markowitz": {k: float(v) for k, v in w_shrink.items()},
    },
    "val_performance": val_df.round(4).to_dict(orient="records"),
    "test_performance": {
        "threshold": best_threshold,
        "copyable_pnl": test_result["copyable_pnl"],
        "copyable_roi": test_result["copyable_roi"],
        "total_pnl": test_result["total_pnl"],
        "trades": test_result["trades"],
        "firing_rate": test_result["firing_rate"],
        "all_trades_copyable_pnl": all_result["copyable_pnl"],
        "all_trades_copyable_roi": all_result["copyable_roi"],
    },
}

out_path = Path("stage1_experimental_result.json")
with open(out_path, "w") as f:
    json.dump(output, f, indent=2, default=str)
print(f"Saved -> {out_path.resolve()}")


Saved -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_experimental_result.json


## TODO: Next Iterations


# TODO: Next Iterations

### Short-term
- [ ] **Full VWAP** - per-trade rolling VWAP (numba-accelerated like _twopass_impl.py)
- [ ] **Aggregated quality wallet volume** - sum all quality wallet volume in window, not just nearest
- [ ] **Top buyer position signal** - track cumulative positions of top N buyers per market
- [ ] **Disagreement signal** - divergence between top buyers (some buying YES, others NO)
- [ ] **Walk-forward cross-validation** - replace single val split

### Medium-term
- [ ] **Deflated Sharpe Ratio** (Bailey et al. 2014) - correct for multiple-signal testing
- [ ] **Non-linear combination** - shallow gradient-boosted ensemble over raw signals
- [ ] **Rolling IC estimation** - re-estimate weights on expanding monthly windows
- [ ] **Calibration layers** - port price_bucket and consensus scores from signal/scorer.py
- [ ] **Execution tape integration** - feed into backtest/execution_tape.py with slippage & latency

### Long-term
- [ ] **Meta-signal from wallet groups** - use polymarket_analysis.copy_groups as signal input
- [ ] **Regime detection** - adjust signal weights per market regime
- [ ] **Full portfolio backtest** - multi-wallet, multi-market with Kelly sizing & risk limits
